# Chain-of-Thought - Forcing the LLM to "Show Its Work"

Hello everyone. Let's dive into another impactful prompting technique: **Chain-of-Thought (CoT) Prompting**.

The core idea is to counteract the model's natural tendency to "rush" to an answer. For any problem that requires logic, arithmetic, or multi-step reasoning, forcing the model to slow down and "show its work" dramatically improves its accuracy. This is particularly effective when working with lower-end LLMs (less reasoning capabilities).

This is done by explicitly instructing the model to think step-by-step. This simple instruction creates a "chain" of reasoning where each thought builds on the last, leading to a more reliable conclusion.

## The Task and Helper Functions

First, let's set up our helper functions and context data.

In [1]:
import inspect
import litellm
from IPython.display import display, Markdown
from textwrap import dedent
#from dotenv import load_dotenv

#load_dotenv()

# --- Configuration & Context ---
MODEL_NAME = "ollama/qwen2.5:3b"
MAX_TOKENS_DEFAULT = 500 # More tokens for reasoning + code

# --- The "messy" code to be refactored ---
def get_user_initials(user_data_list):
    initials_list = []
    for user in user_data_list:
        if 'name' in user and len(user['name']) > 0:
            parts = user['name'].split(' ')
            initials = ''
            for part in parts:
                initials += part[0].upper()
            initials_list.append(initials)
    return initials_list

CODE_TO_REFACTOR = inspect.getsource(get_user_initials)

def get_completion(
    prompt,
    model=MODEL_NAME,
    max_tokens=MAX_TOKENS_DEFAULT,
    **kwargs
):
    parsed_messages = []

    if type(prompt) is str:
        parsed_messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        parsed_messages = prompt

    response = litellm.completion(
        model=model,
        messages=parsed_messages,
        max_tokens=max_tokens,
        **kwargs
    )

    return response.choices[0].message.content


print("Setup complete. Helper functions and code context are ready.")

Setup complete. Helper functions and code context are ready.


## Attempt 1: No Chain-of-Thought

Let's first ask the model for a direct answer without guiding its thought process. This demonstrates the model's default "fast thinking" mode, which often produces only the final result and does not provide much information into how the model tackled the problem.

In [4]:
prompt_no_cot = [
    {
        "role": "user",
        "content": dedent(f"""
        Please refactor the following Python code:

        ```python
        {CODE_TO_REFACTOR}
        ```
        """)
    }
]

response_no_cot = get_completion(prompt_no_cot)

In [5]:
display(Markdown(response_no_cot))

Certainly! Here is a refactored version of the code. I've made it more concise by removing the intermediate `initials` variable and directly constructing the initials string. Additionally, I've added type hints and improved the readability of the function.

```python
def get_user_initials(user_data_list: list) -> list:
    initials_list = []
    for user in user_data_list:
        if 'name' in user and user['name']:
            initials = ''.join(part[0].upper() for part in user['name'].split(' '))
            initials_list.append(initials)
    return initials_list
```

Here are a few improvements made:
- The type hints provide better type information to users of the function.
- The `user['name'].split(' ')` call is more readable and efficient.
- The `initials` variable is constructed directly in the `join` method, eliminating the need for an intermediate `initials` variable.

Let me know if you need further assistance!

## Attempt 2: Introducing Chain-of-Thought

For maximum reliability on complex, repeatable tasks, we can use **Chain-of-Thought**, which can become even more powerful when combined with a couple few-shot examples.

In [12]:
prompt_with_cot = [
    {
        "role": "user",
        "content": dedent(f"""
        Please refactor the following Python code:
        Let's think step by step and please provide your reasoning

        While solving the problem, consider, among others, documentation,
        typing, readability, etc.
        
        ```python
        {CODE_TO_REFACTOR}
        ```
        """)
    }
]

response_with_cot = get_completion(prompt_with_cot)

In [13]:
display(Markdown(response_with_cot))

Sure, I'll refactor the provided code to make it more readable, maintainable, and efficient. Let's go through it step by step.

### Step 1: Use `assert` for Input Validation

Since the input `user_data_list` is expected to be a list of dictionaries, we can add an assertion to ensure that `user_data_list` is not empty.

```python
def get_user_initials(user_data_list):
    assert user_data_list, "user_data_list must not be empty"
```

### Step 2: Split the Name Logic into a Separate Function

The logic of splitting the name into initials can be extracted into a separate function, which will make the code more modular and easier to read.

```python
def split_name(name):
    parts = name.split(' ')
    initials = ''.join(part[0].upper() for part in parts)
    return initials
```

### Step 3: Refactor the Main Function

With the `split_name` function in place, we can refactor the main function to use it and return the list of initials.

```python
def get_user_initials(user_data_list):
    assert user_data_list, "user_data_list must not be empty"
    initials_list = []

    for user in user_data_list:
        if 'name' in user and len(user['name']) > 0:
            initials_list.append(split_name(user['name']))

    return initials_list
```

### Final Refactored Code:

```python
def get_user_initials(user_data_list):
    assert user_data_list, "user_data_list must not be empty"
    initials_list = []

    for user in user_data_list:
        if 'name' in user and len(user['name']) > 0:
            initials_list.append(split_name(user['name']))

    return initials_list

def split_name(name):
    parts = name.split(' ')
    initials = ''.join(part[0].upper() for part in parts)
    return initials
```

### Reasoning:
1. **Assertions**: We use an assertion to ensure that `user_data_list` is not empty. This is good for ensuring the function handles invalid inputs gracefully.
2. **Functionality Separation**: By separating the logic of splitting the name into a function (`split_name`), the main function (`get_user_initials`) becomes cleaner and easier to understand. The `get_user_initials` function is now focused on the core logic

In [15]:
# MH use better model to see if it improves this.
# must stop qwen2.5 and start qwen3 outside of this, 
#   don't forget to revert!

MODEL_NAME = "ollama/qwen3"
response_with_cot_and_brains = get_completion(prompt_with_cot)

In [16]:
display(Markdown(response_with_cot_and_brains))

To refactor the given Python code, let's consider a few key aspects: readability, maintainability, and the use of built-in functions where appropriate. Here’s the step-by-step refactoring process along with the reasoning:

1. **Extract the Initials Calculation into a Separate Function:**
   - This will make the function more modular and easier to test.

2. **Use `list comprehension` for Simplification:**
   - Instead of using a `for` loop to append to the `initials_list`, we can use a list comprehension, which will make the code more concise.

3. **Use a `defaultdict` for `name` Handling:**
   - This will simplify handling `None` or missing `'name'` keys.

4. **Use `str.title` for Capitalization:**
   - This will make the code more readable and avoid concatenation.

Let's implement these changes:

```python
from collections import defaultdict

def get_user_initials(user_data_list):
    initials_list = []
    
    # Create a defaultdict to handle missing or empty 'name'
    user_name_map = defaultdict(str)
    user_name_map.update(user['name'] for user in user_data_list if 'name' in user and user['name'])

    # Iterate over the defaultdict and calculate initials
    for name in user_name_map.values():
        initials = ''.join(part[0].upper() for part in name.split(' '))
        initials_list.append(initials)

    return initials_list

# Example usage:
user_data_list = [
    {'name': 'john doe'},
    {'name': 'jane smith'},
    {'name': 'jack'}
]
print(get_user_initials(user_data_list))
```

### Explanation of Changes:

1. **Extracted Initials Calculation:**
   - The logic of calculating initials was moved into a loop, making the function more modular. This way, if we need to change how initials are calculated, we only need to modify one part of the code.

2. **List Comprehension:**
   - Replaced the `for` loop with `user_name_map.values()` which is a list, and then used a list comprehension for appending the initials. This makes the code more readable and efficient.

3. **DefaultDict for `name` Handling:**
   - Using `defaultdict` simplifies the code by automatically handling missing or empty `'name'` keys without needing to check for them explicitly. This